<a href="https://colab.research.google.com/github/ohmp/LLM-Finetuning/blob/main/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quick Start

**Home GitHub Repository:** [LEANN on GitHub](https://github.com/yichuan-w/LEANN)

**Important for Colab users:** Set your runtime type to T4 GPU for optimal performance. Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU.

In [1]:
# install this if you are using colab
! uv pip install leann-core leann-backend-hnsw --no-deps
! uv pip install leann --no-deps
# For Colab environment, we need to set some environment variables
import os

os.environ["LEANN_LOG_LEVEL"] = "INFO"  # Enable more detailed logging

Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 133ms
Prepared 2 packages in 3.85s
Installed 2 packages in 46ms
 + leann-backend-hnsw==0.3.6
 + leann-core==0.3.6
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 84ms
Prepared 1 package in 38ms
Installed 1 package in 1ms
 + leann==0.3.6


In [2]:
from pathlib import Path

INDEX_DIR = Path("./").resolve()
INDEX_PATH = str(INDEX_DIR / "demo.leann")

## Build the index

In [3]:
from leann.api import LeannBuilder

builder = LeannBuilder(backend_name="hnsw")
builder.add_text("C# is a powerful programming language and it is good at game development")
builder.add_text(
    "Python is a powerful programming language and it is good at machine learning tasks"
)
builder.add_text("Machine learning transforms industries")
builder.add_text("Neural networks process complex data")
builder.add_text("Leann is a great storage saving engine for RAG on your MacBook")
builder.build_index(INDEX_PATH)

INFO:leann.embedding_compute:Computing embeddings for 1 texts using SentenceTransformer, model: 'facebook/contriever'
INFO:leann.embedding_compute:Loading and caching optimized SentenceTransformer model: facebook/contriever
INFO:leann.embedding_compute:Using device: cpu
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.), trying network download...


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

INFO:leann.embedding_compute:Model loaded successfully! (network + optimized)
INFO:leann.embedding_compute:Model cached: sentence_transformers_facebook/contriever_cpu_True_optimized
INFO:leann.embedding_compute:Starting embedding computation... (batch_size: 32, manual_tokenize=False)
INFO:leann.embedding_compute:Generated 1 embeddings, dimension: 768
INFO:leann.embedding_compute:Time taken: 0.19249916076660156 seconds
Writing passages: 100%|██████████| 5/5 [00:00<00:00, 14285.78chunk/s]
INFO:leann.embedding_compute:Computing embeddings for 5 texts using SentenceTransformer, model: 'facebook/contriever'
INFO:leann.embedding_compute:Using cached optimized model: facebook/contriever
INFO:leann.embedding_compute:Starting embedding computation... (batch_size: 32, manual_tokenize=False)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:leann.embedding_compute:Generated 5 embeddings, dimension: 768
INFO:leann.embedding_compute:Time taken: 1.243964433670044 seconds


Starting conversion: /content/demo.index -> /content/demo.csr.tmp
[0.00s] Reading Index HNSW header...
[0.00s]   Header read: d=768, ntotal=5
[0.00s] Reading HNSW struct vectors...
  Reading vector (dtype=<class 'numpy.float64'>, fmt='d')... Count=6, Bytes=48
[0.00s]   Read assign_probas (6)
  Reading vector (dtype=<class 'numpy.int32'>, fmt='i')... Count=7, Bytes=28
[0.26s]   Read cum_nneighbor_per_level (7)
  Reading vector (dtype=<class 'numpy.int32'>, fmt='i')... Count=5, Bytes=20
[0.52s]   Read levels (5)
[0.79s]   Probing for compact storage flag...
[0.79s]   Found compact flag: False
[0.79s]   Compact flag is False, reading original format...
[0.79s]   Probing for potential extra byte before non-compact offsets...
[0.79s]   Found and consumed an unexpected 0x00 byte.
  Reading vector (dtype=<class 'numpy.uint64'>, fmt='Q')... Count=6, Bytes=48
[0.79s]   Read offsets (6)
[1.05s]   Attempting to read neighbors vector...
  Reading vector (dtype=<class 'numpy.int32'>, fmt='i')... Co

## Search with real-time embeddings

In [4]:
from leann.api import LeannSearcher

searcher = LeannSearcher(INDEX_PATH)
results = searcher.search("programming languages", top_k=2)
results

[SearchResult(id='0', score=np.float32(0.9873247), text='C# is a powerful programming language and it is good at game development', metadata={}),
 SearchResult(id='1', score=np.float32(0.89235324), text='Python is a powerful programming language and it is good at machine learning tasks', metadata={})]

## Chat with LEANN using retrieved results

In [5]:
from leann.api import LeannChat

llm_config = {
    "type": "hf",
    "model": "Qwen/Qwen3-0.6B",
}

chat = LeannChat(index_path=INDEX_PATH, llm_config=llm_config)
response = chat.ask(
    "Compare the two retrieved programming languages and tell me their advantages.",
    top_k=2,
    llm_kwargs={"max_tokens": 128},
)
response

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

kwargs in HF:  {'max_tokens': 128}


'<think>\n\n</think>\n\nBased on the context provided:\n\n- **C#** is a powerful programming language and is well-suited for game development.\n- **Python** is also a powerful language and is good at machine learning tasks.\n\n**Advantages:**\n- **C#** is ideal for game development due to its strong support for game engines and robust object-oriented programming.\n- **Python** excels in machine learning tasks due to its simplicity and extensive libraries for data science and AI.'